# URTIC cold vs non-cold — full self-contained Conformer pipeline`n
`n
This single notebook contains the complete preprocessing, feature extraction, train-only feature normalization/selection/PCA, Conformer, focal-loss training, development-set Youden-J thresholding, and final test inference code.`n
`n
**Before running:** change `NOTEBOOK_DATA_ROOT` in the final cell to the extracted URTIC folder on this laptop. Run the dependency cell once, then run the complete pipeline cell. This is configured for the **full dataset**—there is no `--limit-per-split` option in its notebook configuration.

In [1]:
# Run once to install required packages

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "numpy>=1.26",
    "pandas>=2.1",
    "scipy>=1.11",
    "librosa>=0.10",
    "soundfile>=0.12",
    "scikit-learn>=1.3",
    "joblib>=1.3",
    "torch>=2.2",
    "tqdm>=4.66"
])

0

In [2]:
"""Reproducible URTIC cold-vs-non-cold classifier.

No test labels are consulted before final reporting.  Run `python
train_urtic_conformer.py --help` for all options.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import math
import os
import random
import shutil
import time
import urllib.request
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

import joblib
import librosa
import numpy as np
import pandas as pd
from scipy import linalg, signal
from scipy.fft import dct
from scipy.ndimage import median_filter
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             precision_recall_fscore_support, roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm

EPS = 1e-10
AUDIO_SUFFIXES = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
SPLIT_ALIASES = {
    "train": {"train", "training"},
    "dev": {"dev", "develop", "development", "valid", "validation", "val"},
    "test": {"test", "testing"},
}
NONCOLD_WORDS = {"noncold", "non_cold", "non-cold", "nocold", "no_cold", "healthy", "control", "normal"}


@dataclass(frozen=True)
class Config:
    sr: int = 8000
    frame_ms: float = 25.0
    hop_ms: float = 10.0
    trim_db: int = 30
    preemphasis: float = 0.97
    n_mfcc: int = 13
    n_mels: int = 40
    max_seconds: float = 10.0
    selected_dim: int = 96
    pca_dim: int = 64
    d_model: int = 128
    heads: int = 4
    layers: int = 3
    dropout: float = 0.20
    focal_gamma: float = 2.0

    @property
    def frame(self) -> int: return round(self.sr * self.frame_ms / 1000)
    @property
    def hop(self) -> int: return round(self.sr * self.hop_ms / 1000)


def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


def get_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    p.add_argument("--data-root", type=Path, required=True, help="Extracted URTIC directory")
    p.add_argument("--output-dir", type=Path, required=True)
    p.add_argument("--dataset-url", default=None, help="Authorised ZIP URL; used only if data-root is absent")
    p.add_argument("--seed", type=int, default=2026)
    p.add_argument("--epochs", type=int, default=30); p.add_argument("--patience", type=int, default=6)
    p.add_argument("--batch-size", type=int, default=16); p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--workers", type=int, default=0); p.add_argument("--limit-per-split", type=int, default=None)
    p.add_argument("--max-seconds", type=float, default=10.0, help="0 means retain all audio")
    return p.parse_args()


def download_if_needed(root: Path, url: str | None) -> None:
    if root.exists(): return
    if not url: raise FileNotFoundError(f"{root} does not exist. Extract authorised URTIC data there or pass --dataset-url.")
    root.parent.mkdir(parents=True, exist_ok=True)
    archive = root.parent / "urtic_download.zip"
    print(f"Downloading authorised dataset archive to {archive} ...")
    urllib.request.urlretrieve(url, archive)  # noqa: S310 - explicit user-supplied URL
    with zipfile.ZipFile(archive) as z: z.extractall(root.parent)
    archive.unlink()
    if not root.exists():
        raise FileNotFoundError(f"Archive extracted, but expected --data-root {root} was not created. Set --data-root to its extracted directory.")


def find_split(root: Path, aliases: set[str]) -> Path | None:
    found = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() in aliases]
    return min(found, key=lambda x: (len(x.parts), str(x).lower())) if found else None


def label_for(path: Path) -> str | None:
    names = {part.lower().replace(" ", "_") for part in path.parts}
    if names & NONCOLD_WORDS: return "noncold"
    return "cold" if "cold" in names else None


def build_manifest(root: Path, limit: int | None) -> pd.DataFrame:
    directories = {name: find_split(root, aliases) for name, aliases in SPLIT_ALIASES.items()}
    missing = [x for x, p in directories.items() if p is None]
    if missing: raise FileNotFoundError(f"Missing split directories: {missing}. Found: {directories}")
    rows: list[dict] = []
    for split, folder in directories.items():
        records = []
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES:
                label = label_for(path)
                # Official challenge test sets may deliberately omit class folders/labels.
                if label is not None or split == "test":
                    rel = path.relative_to(folder)
                    speaker = rel.parts[-3] if len(rel.parts) >= 3 else path.stem.split("_")[0]
                    records.append(dict(audio_path=str(path.resolve()), split=split, label=label, target=(int(label == "cold") if label is not None else np.nan), speaker_id=speaker))
        if limit: records = records[:limit]
        rows.extend(records)
    table = pd.DataFrame(rows)
    if table.empty or set(table.split) != {"train", "dev", "test"}:
        raise RuntimeError("No labelled files in every split. Check the expected directory/class names in README.md.")
    count = table.groupby(["split", "label"]).size().unstack(fill_value=0)
    labelled = count.reindex(index=["train", "dev"], columns=["cold", "noncold"], fill_value=0)
    if (labelled == 0).any().any(): raise RuntimeError(f"Both classes are required in train and development:\n{count}")
    print("Files discovered:\n", count)
    return table


def feature_names(cfg: Config) -> list[str]:
    base = [f"mfcc_{i+1}" for i in range(cfg.n_mfcc)] + [f"delta_{i+1}" for i in range(cfg.n_mfcc)] + [f"delta2_{i+1}" for i in range(cfg.n_mfcc)]
    base += [f"logmel_{i+1}" for i in range(cfg.n_mels)]
    base += ["f0_hz", "f0_mean", "f0_std", "f0_range", "f0_slope", "hnr_db", "cpps", "jitter_local", "shimmer_local", "f1_hz", "f2_hz", "f3_hz", "spectral_tilt", "spectral_flux", "spectral_entropy", "lf_hf_ratio"]
    nasal = ["nre", "noer", "antiformant_hz", "antiformant_depth_db", "nasal_coupling_strength", "nfd_hz", "nfd_velocity", "nfd_acceleration", "nasality_persistence"]
    temporal = ["modulation_spectrum", "dct_modulation_spectrum", "modulation_energy", "modulation_entropy"]
    return base + nasal + temporal


def preprocess(path: str, cfg: Config) -> np.ndarray:
    y, old_sr = librosa.load(path, sr=None, mono=True)
    if len(y) == 0: raise ValueError("empty audio")
    if old_sr != cfg.sr: y = librosa.resample(y, orig_sr=old_sr, target_sr=cfg.sr)
    y = np.asarray(y, np.float32); y -= y.mean()
    intervals = librosa.effects.split(y, top_db=cfg.trim_db, frame_length=cfg.frame, hop_length=cfg.hop)
    if len(intervals): y = np.concatenate([y[a:b] for a, b in intervals])
    if not len(y): y = np.zeros(cfg.frame, dtype=np.float32)
    y = .98 * y / max(np.max(np.abs(y)), EPS)
    y = signal.lfilter([1.0, -cfg.preemphasis], [1.0], y).astype(np.float32)
    if cfg.max_seconds > 0: y = y[:round(cfg.max_seconds * cfg.sr)]
    return np.pad(y, (0, max(0, cfg.frame - len(y))))


def align(a: np.ndarray, n: int) -> np.ndarray:
    """Make a frame feature n long without inventing a time offset."""
    if len(a) >= n: return a[:n]
    return np.pad(a, (0, n - len(a)), mode="edge") if len(a) else np.zeros(n)


def autocorr_features(frames: np.ndarray, cfg: Config) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n = len(frames); f0 = np.zeros(n); hnr = np.zeros(n); jitter = np.zeros(n); shimmer = np.zeros(n)
    lo, hi = max(1, cfg.sr // 400), min(cfg.frame - 2, cfg.sr // 60)
    periods = np.zeros(n); amps = np.sqrt(np.mean(frames * frames, axis=1) + EPS)
    for i, frame in enumerate(frames):
        x = frame - frame.mean(); ac = signal.correlate(x, x, mode="full", method="fft")[len(x)-1:]
        section = ac[lo:hi]
        if ac[0] > EPS and len(section):
            j = int(section.argmax()) + lo; r = np.clip(ac[j] / ac[0], 0, .999999)
            if r > .25: periods[i] = j; f0[i] = cfg.sr / j; hnr[i] = 10 * np.log10((r + EPS) / (1-r + EPS))
    voiced = periods > 0
    for i in range(1, n):
        if voiced[i] and voiced[i-1]:
            jitter[i] = abs(periods[i] - periods[i-1]) / max((periods[i] + periods[i-1]) / 2, 1)
            shimmer[i] = abs(amps[i] - amps[i-1]) / max((amps[i] + amps[i-1]) / 2, EPS)
    return f0, hnr, jitter, shimmer


def lpc_formants(frame: np.ndarray, cfg: Config) -> tuple[float, float, float]:
    x = frame * np.hamming(len(frame)); order = min(12, len(x) - 2)
    ac = signal.correlate(x, x, mode="full")[len(x)-1:len(x)+order]
    try: coeff = linalg.solve_toeplitz(ac[:-1], ac[1:])
    except Exception: return (0., 0., 0.)
    roots = np.roots(np.r_[1., -coeff]); roots = roots[np.imag(roots) >= .01]
    hz = np.sort(np.arctan2(np.imag(roots), np.real(roots)) * cfg.sr / (2*np.pi))
    hz = hz[(hz > 90) & (hz < cfg.sr/2 - 100)]
    return tuple(np.pad(hz[:3], (0, max(0, 3-len(hz)))))


def _slope(a: np.ndarray) -> float:
    valid = a[a > 0]
    return float(np.polyfit(np.arange(len(valid)), valid, 1)[0]) if len(valid) > 1 else 0.


def extract_features(path: str, cfg: Config) -> np.ndarray:
    y = preprocess(path, cfg); n_fft = 256
    stft = librosa.stft(y, n_fft=n_fft, hop_length=cfg.hop, win_length=cfg.frame, center=False)
    mag = np.abs(stft) + EPS; power = mag**2; n = mag.shape[1]
    # Match signal frames to STFT frames exactly.
    padded = np.pad(y, (0, max(0, (n-1)*cfg.hop+cfg.frame-len(y))))
    frames = librosa.util.frame(padded, frame_length=cfg.frame, hop_length=cfg.hop).T[:n] * np.hanning(cfg.frame)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(power), sr=cfg.sr, n_mfcc=cfg.n_mfcc).T
    delta_width = min(5, n if n % 2 else n-1)
    if delta_width < 3: d1 = np.zeros_like(mfcc); d2 = np.zeros_like(mfcc)
    else: d1 = librosa.feature.delta(mfcc.T, width=delta_width).T; d2 = librosa.feature.delta(mfcc.T, order=2, width=delta_width).T
    logmel = librosa.power_to_db(librosa.feature.melspectrogram(S=power, sr=cfg.sr, n_mels=cfg.n_mels)).T
    f0, hnr, jitter, shimmer = autocorr_features(frames, cfg)
    f0_valid = f0[f0 > 0]; stats = np.array([f0_valid.mean() if len(f0_valid) else 0., f0_valid.std() if len(f0_valid) else 0., np.ptp(f0_valid) if len(f0_valid) else 0., _slope(f0)])
    freq = librosa.fft_frequencies(sr=cfg.sr, n_fft=n_fft); db = 20*np.log10(mag)
    # CPPS: cepstral peak above a median baseline in the plausible pitch-quefrency range.
    cep = np.fft.irfft(np.log(mag), axis=0); qlo, qhi = max(1, cfg.sr//400), min(cep.shape[0], cfg.sr//60)
    cpps = np.max(cep[qlo:qhi], axis=0) - np.median(cep[qlo:qhi], axis=0) if qhi > qlo else np.zeros(n)
    formants = np.array([lpc_formants(f, cfg) for f in frames])
    p = power / np.maximum(power.sum(axis=0, keepdims=True), EPS)
    entropy = -(p*np.log(p)).sum(axis=0) / math.log(len(freq))
    flux = np.r_[0., np.sqrt(np.mean(np.diff(mag, axis=1)**2, axis=0))]
    tilt = np.array([np.polyfit(freq, db[:, i], 1)[0] for i in range(n)])
    low = power[(freq >= 80) & (freq < 1000)].sum(axis=0); high = power[(freq >= 1000) & (freq < 3800)].sum(axis=0)
    lf_hf = np.log((low + EPS) / (high + EPS))
    # Nasality: NRE = energy around the first nasal-resonance band; NOER = nasal/oral energy.
    nasal = power[(freq >= 250) & (freq <= 650)].sum(axis=0); oral = power[(freq >= 650) & (freq <= 3000)].sum(axis=0)
    full = power.sum(axis=0); nre = nasal/(full+EPS); noer = np.log((nasal+EPS)/(oral+EPS))
    anti_hz = np.zeros(n); anti_depth = np.zeros(n); nfd = np.zeros(n)
    region = np.where((freq >= 350) & (freq <= 2500))[0]
    nasal_region = np.where((freq >= 250) & (freq <= 1000))[0]
    for i in range(n):
        smooth = median_filter(db[:, i], size=5)
        j = region[np.argmin(smooth[region])]; anti_hz[i] = freq[j]; anti_depth[i] = np.median(smooth[region])-smooth[j]
        peak = nasal_region[np.argmax(smooth[nasal_region])]; nfd[i] = freq[peak] - 500.0
    nfd_v = np.gradient(nfd) / (cfg.hop_ms / 1000); nfd_a = np.gradient(nfd_v) / (cfg.hop_ms / 1000)
    coupling = nre * (1/(1+np.exp(-noer))) * (anti_depth/(anti_depth+6+EPS))
    threshold = np.median(nre); persistence = np.convolve((nre > threshold).astype(float), np.ones(15)/15, mode="same")
    # Temporal modulation measures, interpolated to every frame from a log-Mel envelope.
    envelope = logmel.mean(axis=1) - logmel.mean()
    mod = np.abs(np.fft.rfft(envelope)); mod_freq = np.fft.rfftfreq(n, d=cfg.hop_ms/1000)
    speech_mod = mod[(mod_freq >= .5) & (mod_freq <= 16)]
    ms = np.full(n, speech_mod.mean() if len(speech_mod) else 0.)
    dct_coeff = np.abs(dct(envelope * np.hanning(n), type=2, norm="ortho"))
    dct_ms = np.full(n, dct_coeff[1:min(len(dct_coeff), 17)].mean() if len(dct_coeff)>1 else 0.)
    mod_energy = np.full(n, np.mean(speech_mod**2) if len(speech_mod) else 0.)
    mp = speech_mod / max(speech_mod.sum(), EPS); mod_entropy = np.full(n, -(mp*np.log(mp+EPS)).sum() if len(mp) else 0.)
    cols = [mfcc, d1, d2, logmel, f0[:,None], np.tile(stats, (n,1)), hnr[:,None], cpps[:,None], jitter[:,None], shimmer[:,None], formants,
            tilt[:,None], flux[:,None], entropy[:,None], lf_hf[:,None], nre[:,None], noer[:,None], anti_hz[:,None], anti_depth[:,None], coupling[:,None], nfd[:,None], nfd_v[:,None], nfd_a[:,None], persistence[:,None], ms[:,None], dct_ms[:,None], mod_energy[:,None], mod_entropy[:,None]]
    x = np.concatenate(cols, axis=1).astype(np.float32)
    return np.nan_to_num(x, nan=0., posinf=0., neginf=0.)


def cache_file(audio: str, cache: Path, fingerprint: str) -> Path:
    p = Path(audio); identity = f"{p.resolve()}|{p.stat().st_size}|{p.stat().st_mtime_ns}|{fingerprint}"
    return cache / (hashlib.sha256(identity.encode()).hexdigest() + ".npz")


def cache_all(manifest: pd.DataFrame, cache: Path, cfg: Config) -> pd.DataFrame:
    cache.mkdir(parents=True, exist_ok=True); fingerprint = hashlib.sha256(json.dumps(asdict(cfg), sort_keys=True).encode()).hexdigest()[:16]
    saved, failures = [], []
    expected = len(feature_names(cfg))
    for row in tqdm(manifest.itertuples(), total=len(manifest), desc="Extracting features"):
        out = cache_file(row.audio_path, cache, fingerprint)
        try:
            valid = out.exists() and np.load(out, allow_pickle=False)["x"].shape[1] == expected
            if not valid:
                x = extract_features(row.audio_path, cfg)
                if x.shape[1] != expected: raise ValueError(f"Expected {expected} features, got {x.shape}")
                tmp = out.with_suffix(".tmp.npz"); np.savez_compressed(tmp, x=x, fingerprint=fingerprint); tmp.replace(out)
            saved.append(str(out.resolve()))
        except Exception as e: failures.append({"audio_path": row.audio_path, "error": repr(e)}); saved.append(None)
    result = manifest.copy(); result["cache_path"] = saved
    if failures: pd.DataFrame(failures).to_csv(cache.parent / "cache_failures.csv", index=False)
    return result.dropna(subset=["cache_path"]).copy()


def load(path: str) -> np.ndarray:
    with np.load(path, allow_pickle=False) as z: return z["x"].astype(np.float32)


def fit_transforms(train: pd.DataFrame, out: Path, cfg: Config) -> tuple[StandardScaler, np.ndarray, IncrementalPCA | None]:
    scaler = StandardScaler(); mean = None; sq = None; count = 0
    for p in tqdm(train.cache_path, desc="Fitting train-only scaler"):
        x = load(p); scaler.partial_fit(x)
        mean = x.sum(0) if mean is None else mean+x.sum(0); sq = (x*x).sum(0) if sq is None else sq+(x*x).sum(0); count += len(x)
    variance = sq/count - (mean/count)**2
    keep = np.argsort(variance)[::-1][:min(cfg.selected_dim, len(variance))]
    np.savez(out/"scaler.npz", mean=scaler.mean_, scale=scaler.scale_); np.savez(out/"selection.npz", indices=keep, raw_variance=variance)
    pca_dim = min(cfg.pca_dim, len(keep))
    pca = IncrementalPCA(n_components=pca_dim, batch_size=4096) if pca_dim < len(keep) else None
    if pca:
        for p in tqdm(train.cache_path, desc="Fitting train-only PCA"):
            x = (load(p)-scaler.mean_) / np.maximum(scaler.scale_, 1e-6); pca.partial_fit(x[:, keep])
        joblib.dump(pca, out/"pca.joblib")
    return scaler, keep, pca


class CachedDataset(Dataset):
    def __init__(self, table: pd.DataFrame, scaler: StandardScaler, keep: np.ndarray, pca: IncrementalPCA | None): self.t, self.scaler, self.keep, self.pca = table.reset_index(drop=True), scaler, keep, pca
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        r = self.t.iloc[i]; x=(load(r.cache_path)-self.scaler.mean_)/np.maximum(self.scaler.scale_, 1e-6); x=np.clip(x[:, self.keep], -8, 8)
        if self.pca: x=self.pca.transform(x)
        target = -1 if pd.isna(r.target) else int(r.target)
        return torch.from_numpy(x.astype(np.float32)), target, r.audio_path


def collate(batch):
    xs, ys, paths = zip(*batch); lengths = torch.tensor([len(x) for x in xs]); x=torch.zeros(len(xs), int(lengths.max()), xs[0].shape[1])
    for i, item in enumerate(xs): x[i,:len(item)] = item
    return x, torch.tensor(ys), lengths, list(paths)


class ConformerBlock(nn.Module):
    def __init__(self, d: int, heads: int, dropout: float):
        super().__init__(); hidden=d*4
        self.ff1=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,hidden),nn.SiLU(),nn.Dropout(dropout),nn.Linear(hidden,d))
        self.attn_norm=nn.LayerNorm(d); self.attn=nn.MultiheadAttention(d,heads,dropout=dropout,batch_first=True)
        self.conv_norm=nn.LayerNorm(d); self.conv=nn.Sequential(nn.Conv1d(d,2*d,1),nn.GLU(1),nn.Conv1d(d,d,15,padding=7,groups=d),nn.BatchNorm1d(d),nn.SiLU(),nn.Conv1d(d,d,1),nn.Dropout(dropout))
        self.ff2=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,hidden),nn.SiLU(),nn.Dropout(dropout),nn.Linear(hidden,d)); self.norm=nn.LayerNorm(d)
    def forward(self, x, pad):
        x=x+.5*self.ff1(x); a=self.attn_norm(x); x=x+self.attn(a,a,a,key_padding_mask=pad,need_weights=False)[0]
        x=x+self.conv(self.conv_norm(x).transpose(1,2)).transpose(1,2); return self.norm(x+.5*self.ff2(x))


class ColdConformer(nn.Module):
    def __init__(self, dim: int, cfg: Config):
        super().__init__(); self.pool=4; self.inp=nn.Sequential(nn.Linear(dim,cfg.d_model),nn.LayerNorm(cfg.d_model),nn.Dropout(cfg.dropout))
        self.blocks=nn.ModuleList([ConformerBlock(cfg.d_model,cfg.heads,cfg.dropout) for _ in range(cfg.layers)])
        self.head=nn.Sequential(nn.Linear(cfg.d_model,cfg.d_model),nn.SiLU(),nn.Dropout(cfg.dropout),nn.Linear(cfg.d_model,2))
    def forward(self,x,lengths):
        x=self.inp(x)[:,::self.pool]; lengths=(lengths+self.pool-1)//self.pool; pad=torch.arange(x.shape[1],device=x.device)[None]>=lengths[:,None]
        for block in self.blocks: x=block(x,pad)
        valid=(~pad).unsqueeze(-1); return self.head((x*valid).sum(1)/lengths[:,None].clamp(min=1))


class FocalLoss(nn.Module):
    def __init__(self,gamma): super().__init__(); self.gamma=gamma
    def forward(self, logits, target):
        ce=F.cross_entropy(logits,target,reduction="none"); return ((1-torch.exp(-ce))**self.gamma*ce).mean()


@torch.no_grad()
def predict(model, loader, device):
    model.eval(); ys=[]; probs=[]; paths=[]
    for x,y,l,p in loader:
        probs.extend(torch.softmax(model(x.to(device),l.to(device)),1)[:,1].cpu().numpy()); ys.extend(y.numpy()); paths.extend(p)
    return np.asarray(ys),np.asarray(probs),paths


def threshold_from_dev(y, p) -> tuple[float, float]:
    fpr,tpr,thresholds=roc_curve(y,p); j=tpr-fpr; i=int(np.argmax(j)); return float(thresholds[i]),float(j[i])


def metrics(y, p, threshold):
    pred=(p>=threshold).astype(int); recalls=precision_recall_fscore_support(y,pred,labels=[0,1],zero_division=0)[1]
    out={"uar":float(recalls.mean()),"balanced_accuracy":float(balanced_accuracy_score(y,pred)),"accuracy":float(accuracy_score(y,pred)),"recall_noncold":float(recalls[0]),"recall_cold":float(recalls[1])}
    if len(np.unique(y)) == 2: out["roc_auc"]=float(roc_auc_score(y,p))
    return out,pred


def main() -> None:
    args=get_args(); seed_everything(args.seed); cfg=Config(max_seconds=args.max_seconds)
    args.output_dir.mkdir(parents=True,exist_ok=True); download_if_needed(args.data_root,args.dataset_url)
    with open(args.output_dir/"run_config.json","w") as f: json.dump({"args":vars(args)|{"data_root":str(args.data_root),"output_dir":str(args.output_dir)},"features":asdict(cfg)},f,indent=2,default=str)
    names=feature_names(cfg); (args.output_dir/"feature_schema.json").write_text(json.dumps({"names":names,"dimension":len(names),"notes":{"NRE":"250-650 Hz energy / total energy","NOER":"log nasal (250-650 Hz) / oral (650-3000 Hz) energy","NFD":"dominant 250-1000 Hz resonance minus 500 Hz","CPPS":"short-time cepstral-peak prominence estimate"}},indent=2))
    manifest=build_manifest(args.data_root,args.limit_per_split); manifest=cache_all(manifest,args.output_dir/"feature_cache",cfg); manifest.to_csv(args.output_dir/"manifest.csv",index=False)
    splits={s:manifest[manifest.split.eq(s)].copy() for s in ("train","dev","test")}
    scaler,keep,pca=fit_transforms(splits["train"],args.output_dir,cfg); datasets={s:CachedDataset(t,scaler,keep,pca) for s,t in splits.items()}
    count=np.bincount(splits["train"].target,minlength=2); weights=1/count[splits["train"].target.to_numpy()]; sampler=WeightedRandomSampler(weights,len(weights),replacement=True,generator=torch.Generator().manual_seed(args.seed))
    loaders={"train":DataLoader(datasets["train"],batch_size=args.batch_size,sampler=sampler,num_workers=args.workers,collate_fn=collate),"dev":DataLoader(datasets["dev"],batch_size=args.batch_size,shuffle=False,num_workers=args.workers,collate_fn=collate),"test":DataLoader(datasets["test"],batch_size=args.batch_size,shuffle=False,num_workers=args.workers,collate_fn=collate)}
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); model=ColdConformer(min(cfg.pca_dim,len(keep)) if pca else len(keep),cfg).to(device); opt=torch.optim.AdamW(model.parameters(),lr=args.lr,weight_decay=1e-3); loss_fn=FocalLoss(cfg.focal_gamma)
    best=-1.; stale=0; history=[]
    for epoch in range(1,args.epochs+1):
        model.train(); losses=[]
        for x,y,l,_ in loaders["train"]:
            opt.zero_grad(set_to_none=True); loss=loss_fn(model(x.to(device),l.to(device)),y.to(device)); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),2.); opt.step(); losses.append(loss.item())
        yd,dev_prob,_=predict(model,loaders["dev"],device); threshold,j=threshold_from_dev(yd,dev_prob); score,_=metrics(yd,dev_prob,threshold); record={"epoch":epoch,"loss":float(np.mean(losses)),"threshold":threshold,"youden_j":j,**score}; history.append(record); print(record)
        if score["uar"] > best+1e-6:
            best=score["uar"]; stale=0; torch.save({"state_dict":model.state_dict(),"feature_dim":model.inp[0].in_features,"config":asdict(cfg)},args.output_dir/"best_model.pt")
        else:
            stale+=1
            if stale>=args.patience: break
    pd.DataFrame(history).to_csv(args.output_dir/"history.csv",index=False)
    state=torch.load(args.output_dir/"best_model.pt",map_location=device,weights_only=False); model.load_state_dict(state["state_dict"])
    yd,dev_prob,_=predict(model,loaders["dev"],device); threshold,j=threshold_from_dev(yd,dev_prob); json.dump({"threshold":threshold,"youden_j":j,"selected_on":"development_only"},open(args.output_dir/"dev_threshold.json","w"),indent=2)
    yt,pt,paths=predict(model,loaders["test"],device); pred=(pt>=threshold).astype(int)
    pd.DataFrame({"audio_path":paths,"cold_probability":pt,"prediction":pred,"threshold":threshold,"target_if_available":yt}).to_csv(args.output_dir/"test_predictions.csv",index=False)
    if np.all(yt >= 0):
        mt,_=metrics(yt,pt,threshold); json.dump(mt,open(args.output_dir/"test_metrics.json","w"),indent=2); print("Final held-out test metrics:",json.dumps(mt,indent=2)); print(classification_report(yt,pred,target_names=["noncold","cold"],zero_division=0))
    else:
        (args.output_dir/"test_metrics.json").write_text(json.dumps({"status":"not computed: test labels are unavailable"},indent=2)); print("Saved test probabilities; no metrics computed because test labels are blinded.")


# --- Notebook configuration: change DATA_ROOT if needed, then run this cell. ---
NOTEBOOK_DATA_ROOT = r"E:\Final Year Project\dataset\URTIC"
NOTEBOOK_OUTPUT_DIR = r"outputs\urtic_full_run"
NOTEBOOK_EPOCHS = 30
NOTEBOOK_PATIENCE = 6
NOTEBOOK_BATCH_SIZE = 16

import sys
sys.argv = [
    "train_urtic_conformer.py",
    "--data-root", NOTEBOOK_DATA_ROOT,
    "--output-dir", NOTEBOOK_OUTPUT_DIR,
    "--epochs", str(NOTEBOOK_EPOCHS),
    "--patience", str(NOTEBOOK_PATIENCE),
    "--batch-size", str(NOTEBOOK_BATCH_SIZE),
]
main()


Files discovered:
 label  cold  noncold
split               
dev    1011     8585
test    894     8655
train   970     8532


Extracting features:  64%|██████▍   | 18368/28647 [00:52<00:25, 399.54it/s]c:\Users\Srichaitanya Panda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Srichaitanya Panda\AppData\Local\Temp\ipykernel_14172\1210416183.py:153: UserWarning: PySoundFile failed. Trying audioread instead.
  y, old_sr = librosa.load(path, sr=None, mono=True)
c:\Users\Srichaitanya Panda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Fitting train-only PCA: 100%|██████████| 9502/9502 [02:34<00:00, 61.45it/s]


{'epoch': 1, 'loss': 0.10274217698114739, 'threshold': 0.24506321549415588, 'youden_j': 0.3296955760631194, 'uar': 0.6648477880315596, 'balanced_accuracy': 0.6648477880315596, 'accuracy': 0.5338196977592496, 'recall_noncold': 0.4988350419384902, 'recall_cold': 0.8308605341246291, 'roc_auc': 0.705211453139418}
{'epoch': 2, 'loss': 0.06394797284469661, 'threshold': 0.11122999340295792, 'youden_j': 0.3591865297201427, 'uar': 0.6795932648600713, 'balanced_accuracy': 0.6795932648600713, 'accuracy': 0.5726941115164148, 'recall_noncold': 0.5441519105312209, 'recall_cold': 0.8150346191889218, 'roc_auc': 0.7255654943800856}
{'epoch': 3, 'loss': 0.036602021060940446, 'threshold': 0.07376556843519211, 'youden_j': 0.27712243605521, 'uar': 0.638561218027605, 'balanced_accuracy': 0.638561218027605, 'accuracy': 0.5874934861907244, 'recall_noncold': 0.5738583410997204, 'recall_cold': 0.7032640949554896, 'roc_auc': 0.6773312758168994}
{'epoch': 4, 'loss': 0.023705359827609487, 'threshold': 0.0279810391